David Beas, Nick Garcia
SVM Linear
Dataset: https://www.kaggle.com/datasets/prince7489/online-learning-platform-usage-dataset

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Load the dataset
df = pd.read_csv('online_learning_platform_usage_dataset.csv')

# Drop the User ID column, as it provides no mathematical value to the model
df = df.drop('user_id', axis=1)

# Data Cleaning: Encode string/text categorical variables into integers
# SVM algorithms require purely numerical data
le_age = LabelEncoder()
df['age_group'] = le_age.fit_transform(df['age_group'])

le_role = LabelEncoder()
df['role'] = le_role.fit_transform(df['role'])

le_platform = LabelEncoder()
df['primary_platform'] = le_platform.fit_transform(df['primary_platform'])

le_device = LabelEncoder()
df['learning_device'] = le_device.fit_transform(df['learning_device'])

le_content = LabelEncoder()
df['content_type_preference'] = le_content.fit_transform(df['content_type_preference'])

# Encode the Target Variable: We are predicting 'learning_purpose'
# (Classes: Academic, Career Growth, Hobby, Skill Upgrade)
le_target = LabelEncoder()
df['learning_purpose'] = le_target.fit_transform(df['learning_purpose'])

df.head()

In [ ]:
# Define Features (Independent) and Target (Dependent) Variables
X = df.drop('learning_purpose', axis=1)
y = df['learning_purpose']

# Split the data into Training (75%) and Testing (25%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Feature Scaling (CRITICAL FOR SVM)
# SVM tries to maximize the margin distance between points. If we don't scale,
# massive variables (like completion percentage) will break the geometry.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Linear SVM using OvO (One-vs-One) strategy
# OvO builds a separate classifier for every pair of classes.
svm_ovo = SVC(kernel='linear', decision_function_shape='ovo', random_state=42)
svm_ovo.fit(X_train_scaled, y_train)
y_pred_ovo = svm_ovo.predict(X_test_scaled)

print("--- Support Vector Machine: Linear Kernel (OvO Strategy) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ovo):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_ovo, average='weighted', zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_ovo, average='weighted', zero_division=0):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_ovo, average='weighted', zero_division=0):.3f}")

In [ ]:
# Train Linear SVM using OvA (One-vs-Rest / One-vs-All) strategy
# OvA builds one classifier per class to separate it from all the rest combined.
svm_ova = SVC(kernel='linear', decision_function_shape='ovr', random_state=42)
svm_ova.fit(X_train_scaled, y_train)
y_pred_ova = svm_ova.predict(X_test_scaled)

print("--- Support Vector Machine: Linear Kernel (OvA Strategy) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ova):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_ova, average='weighted', zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_ova, average='weighted', zero_division=0):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_ova, average='weighted', zero_division=0):.3f}")

In [ ]:
# User Input Loop for New Predictions
def predict_learning_purpose():
    while True:
        try:
            print("\n--- Predict User Learning Purpose ---")
            # Get user inputs for a new data point
            age = int(input("Enter Age Group (0='18-24', 1='25-34', 2='35-44', 3='45+', 4='Under 18'): "))
            role = int(input("Enter Role (0=Freelancer, 1=Other, 2=Student, 3=Working Professional): "))
            platform = int(input("Enter Platform (0=Coursera, 1=Khan Academy, 2=Skillshare, 3=Udemy, 4=YouTube, 5=edX): "))
            device = int(input("Enter Device (0=Desktop, 1=Laptop, 2=Mobile, 3=Tablet): "))
            hours = int(input("Enter Hours Per Week: "))
            content = int(input("Enter Content Preference (0=Mixed, 1=Text, 2=Video): "))
            completion = int(input("Enter Completion Rate % (0-100): "))
            satisfaction = int(input("Enter Satisfaction Score (1-10): "))

            # Create dataframe and scale it
            new_data = pd.DataFrame([[age, role, platform, device, hours, content, completion, satisfaction]],
                                    columns=X.columns)
            new_data_scaled = scaler.transform(new_data)

            # Predict using our OvO model
            pred = svm_ovo.predict(new_data_scaled)[0]
            purpose = le_target.inverse_transform([pred])[0]

            print("\n-----------------------------------------")
            print(f"PREDICTION: This user is most likely taking courses for: {purpose}")
            print("-----------------------------------------")

            run_again = input("Would you like to test another user? (yes/no): ")
            if run_again.lower() != 'yes':
                break
        except ValueError:
            print("Invalid input. Please enter numbers only.")

predict_learning_purpose()

For this analysis, we used a Support Vector Machine (SVM) model with a Linear kernel to classify an online learner's primary motive—whether they are studying for career growth, academic requirements, skill upgrades, or purely as a hobby. We relied on a dataset tracking user device preferences, study hours, roles, and satisfaction metrics. Because our target had four possible outcomes, we justified using and comparing two distinct multi-class strategies: One-vs-One (OvO) and One-vs-All (OvA).

OvO broke the problem down by training a separate model for every possible pair of classes (creating 6 micro-comparisons). In contrast, OvA created just one broad model per class to separate it from all the others combined (creating 4 comparisons). We found that both strategies yielded virtually identical reliability metrics for this dataset. However, testing both was a valuable justification step: it confirmed that the boundaries between our learner profiles are relatively linear and didn't strictly require the highly granular pair-by-pair evaluation of OvO to make a decision. From a business standpoint, having a fast, linear OvA model allows management to rapidly predict why a new demographic is using our platform, helping us tailor marketing campaigns or suggest targeted content pathways specifically to hobbyists versus career professionals.